# What is a British racecourse?

## Introduction

Before counting British racecourses, we first need to establish **what a racecourse actually is**.

That sounds straightforward, but the source data makes the distinction less obvious than it first appears. Some racing venues appear under more than one label. For example, the database distinguishes between `Kempton` and `Kempton (AW)`, and between `Lingfield` and `Lingfield (AW)`.

Those labels should not automatically be treated as separate racecourses.

Authoritative racing sources describe Kempton Park and Lingfield Park as individual racecourses containing different racing tracks or surfaces. In particular, both venues accommodate all-weather racing alongside other forms of racing. This means a source-data label may identify a **particular track or racing configuration within a racecourse**, rather than a separate physical venue.

This study therefore begins with a more fundamental question:

> **When British racing refers to a racecourse, what exactly is being counted?**

We will first examine how racing authorities use terms such as **racecourse**, **course**, **track**, **venue** and **fixture**, and compare that terminology with the identities represented in the Inside Rails database.

Only once that relationship is understood will we attempt to count British racecourses.

The aim is to produce a reader-facing definition that reflects how British racing is actually organised, rather than simply counting distinct source labels and assuming they represent distinct places.


In [1]:
from pathlib import Path

import pandas as pd

from inside_rails.source_sqlite import connect_read_only


# Use the accepted immutable Database v3 release documented for study work.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATABASE_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "database"
    / "releases"
    / "inside_rails_v3.sqlite3"
)

assert DATABASE_PATH.exists(), f"Database not found: {DATABASE_PATH}"


# Examine how the governed race-level data represents Kempton and Lingfield.
#
# These are useful diagnostic cases because external evidence establishes that
# each is one racecourse containing more than one racing track/surface.
#
# We are not yet merging labels or defining a general racecourse identity rule.
# The purpose here is simply to establish what distinctions the database labels
# are currently encoding.
query = """
SELECT
    candidate_course_label,
    COUNT(*) AS races,
    COUNT(DISTINCT raw_date) AS race_dates,
    MIN(raw_date) AS earliest_date,
    MAX(raw_date) AS latest_date
FROM view_reconciled_race_occurrences
WHERE candidate_jurisdiction = 'Great Britain'
  AND candidate_course_label IN (
      'Kempton',
      'Kempton (AW)',
      'Lingfield',
      'Lingfield (AW)'
  )
GROUP BY candidate_course_label
ORDER BY candidate_course_label
"""

with connect_read_only(DATABASE_PATH) as connection:
    course_label_examples = pd.read_sql_query(query, connection)

course_label_examples

,candidate_course_label,races,race_dates,earliest_date,latest_date
0,Kempton,902,132,2015-01-10,2026-03-23
1,Kempton (AW),5046,662,2015-01-07,2026-05-27
2,Lingfield,1457,248,2015-01-05,2026-05-26
3,Lingfield (AW),4803,742,2015-01-03,2026-05-12


## What the source labels are distinguishing

The governed race-level data contains separate `candidate_course_label` values for:

* `Kempton`
* `Kempton (AW)`
* `Lingfield`
* `Lingfield (AW)`

These are substantial and persistent populations rather than occasional naming anomalies. Kempton has 902 races under the unqualified label and 5,046 under `Kempton (AW)`, while Lingfield has 1,457 and 4,803 respectively.

However, the external racing evidence established earlier shows that these labels do **not** represent four separate racecourses.

Kempton Park is one racecourse containing distinct Jump and all-weather racing tracks, while Lingfield Park is one racecourse containing turf and all-weather tracks.

Therefore:

> **`candidate_course_label` identifies a source-level racing course or track distinction, not necessarily a distinct British racecourse.**

In particular, the `(AW)` suffix is carrying information about the all-weather component of the venue. Removing or merging that distinction would lose useful racing information, but counting each label as a separate racecourse would overstate the number of racecourses.

This gives us an important distinction for the rest of the study:

* **racecourse** — the recognised racing venue;
* **course/track distinction** — a particular racing surface or configuration within that venue;
* **`candidate_course_label`** — the database's source-derived label, which may distinguish those internal course/track differences.

We therefore cannot answer “How many British racecourses are there?” by simply counting distinct `candidate_course_label` values.


In [2]:
from pathlib import Path

import pandas as pd

from inside_rails.source_sqlite import connect_read_only


# Use the accepted immutable Database v3 release documented for studies.
DATABASE_PATH = (
    Path.cwd()
    / "../../../data/processed/database/releases/inside_rails_v3.sqlite3"
).resolve()

assert DATABASE_PATH.exists(), f"Database not found: {DATABASE_PATH}"


# Diagnostic question:
# How does the governed race-level data distinguish Kempton and Lingfield?
#
# We inspect only these two racecourses because external racing evidence has
# already established that each is one racecourse containing distinct racing
# tracks/surfaces. We are not yet defining a general racecourse identity rule.
query = """
SELECT
    candidate_course_label,
    COUNT(*) AS races,
    COUNT(DISTINCT raw_date) AS race_dates,
    MIN(raw_date) AS earliest_date,
    MAX(raw_date) AS latest_date
FROM view_reconciled_race_occurrences
WHERE candidate_jurisdiction = 'Great Britain'
  AND candidate_course_label IN (
      'Kempton',
      'Kempton (AW)',
      'Lingfield',
      'Lingfield (AW)'
  )
GROUP BY candidate_course_label
ORDER BY candidate_course_label
"""

with connect_read_only(DATABASE_PATH) as connection:
    kempton_lingfield = pd.read_sql_query(query, connection)

kempton_lingfield

,candidate_course_label,races,race_dates,earliest_date,latest_date
0,Kempton,902,132,2015-01-10,2026-03-23
1,Kempton (AW),5046,662,2015-01-07,2026-05-27
2,Lingfield,1457,248,2015-01-05,2026-05-26
3,Lingfield (AW),4803,742,2015-01-03,2026-05-12


### What this establishes

The database treats `Kempton` and `Kempton (AW)` as separate course labels, and does the same for `Lingfield` and `Lingfield (AW)`.

These are not trivial or short-lived variants. The all-weather labels account for thousands of races across hundreds of race dates over the full study period.

External racing evidence already establishes that Kempton Park and Lingfield Park are individual racecourses containing distinct racing tracks or surfaces. The database labels therefore represent a finer distinction than racecourse identity.

This means `candidate_course_label` cannot be counted directly as a British racecourse identifier.

### What this does not yet establish

This does not tell us how many of the remaining British course labels represent:

- distinct racecourses;
- alternative tracks or surfaces within the same racecourse;
- historical naming variants;
- other source-specific distinctions.

The next step is therefore to inspect the full set of British `candidate_course_label` values and identify which labels may represent multiple configurations of the same racecourse.

In [3]:
query = """
SELECT
    candidate_course_label,
    COUNT(*) AS races,
    COUNT(DISTINCT raw_date) AS race_dates,
    MIN(raw_date) AS earliest_date,
    MAX(raw_date) AS latest_date
FROM view_reconciled_race_occurrences
WHERE candidate_jurisdiction = 'Great Britain'
GROUP BY candidate_course_label
ORDER BY candidate_course_label
"""

with connect_read_only(DATABASE_PATH) as connection:
    gb_course_labels = pd.read_sql_query(query, connection)

gb_course_labels

,candidate_course_label,races,race_dates,earliest_date,latest_date
0,Aintree,610,87,2015-04-09,2026-05-15
1,Ascot,1816,273,2015-01-17,2026-05-09
2,Ayr,2325,328,2015-01-02,2026-05-20
3,Bangor-on-Dee,985,145,2015-01-06,2026-05-23
4,Bath,1540,216,2015-04-17,2026-05-26
...,...,...,...,...,...
60,Windsor,1935,274,2015-04-13,2026-05-25
61,Wolverhampton (AW),7042,930,2015-01-02,2026-05-18
62,Worcester,1399,194,2015-05-07,2026-05-22
63,Yarmouth,1628,230,2015-08-30,2026-05-20


In [4]:
variant_labels = gb_course_labels[
    gb_course_labels["candidate_course_label"].str.contains(r"\(", regex=True)
].copy()

variant_labels

,candidate_course_label,races,race_dates,earliest_date,latest_date
10,Chelmsford (AW),4310,593,2015-01-11,2026-03-26
28,Kempton (AW),5046,662,2015-01-07,2026-05-27
31,Lingfield (AW),4803,742,2015-01-03,2026-05-12
37,Newcastle (AW),4409,583,2016-05-17,2026-05-19
39,Newmarket (July),1438,208,2015-06-19,2025-08-23
51,Southwell (AW),3816,515,2015-01-01,2026-05-21
61,Wolverhampton (AW),7042,930,2015-01-02,2026-05-18


In [5]:
# Compare each parenthetical label with its possible unqualified counterpart.
#
# This is only a source-label diagnostic. A matching base name does not by
# itself prove that both labels belong to one racecourse; that still requires
# racing-domain evidence.

qualified_pairs = variant_labels.copy()

qualified_pairs["possible_base_label"] = (
    qualified_pairs["candidate_course_label"]
    .str.replace(r"\s*\([^)]*\)$", "", regex=True)
)

all_labels = set(gb_course_labels["candidate_course_label"])

qualified_pairs["base_label_present"] = (
    qualified_pairs["possible_base_label"].isin(all_labels)
)

qualified_pairs[
    [
        "candidate_course_label",
        "possible_base_label",
        "base_label_present",
        "races",
        "race_dates",
    ]
]

,candidate_course_label,possible_base_label,base_label_present,races,race_dates
10,Chelmsford (AW),Chelmsford,False,4310,593
28,Kempton (AW),Kempton,True,5046,662
31,Lingfield (AW),Lingfield,True,4803,742
37,Newcastle (AW),Newcastle,True,4409,583
39,Newmarket (July),Newmarket,True,1438,208
51,Southwell (AW),Southwell,True,3816,515
61,Wolverhampton (AW),Wolverhampton,False,7042,930


### What we found

Five qualified source labels have matching unqualified names:

- `Kempton` / `Kempton (AW)`
- `Lingfield` / `Lingfield (AW)`
- `Newcastle` / `Newcastle (AW)`
- `Newmarket` / `Newmarket (July)`
- `Southwell` / `Southwell (AW)`

External racecourse evidence shows that these pairs do not all mean the same thing.

At Kempton, Lingfield, Newcastle and Southwell, the paired labels distinguish racing tracks or surfaces within a single racecourse. Their source labels therefore need to be collapsed when counting racecourses, while retaining the track distinction for analytical purposes.

Newmarket is different. The Jockey Club explicitly describes the Rowley Mile and July Course as two separate racecourses. `Newmarket` and `Newmarket (July)` therefore represent two racecourse identities rather than merely two track labels within one racecourse.

### Interpretation

A parenthetical qualifier cannot itself determine racecourse identity.

The source label must be interpreted against the real-world racing entity it represents. In particular:

- `(AW)` can identify an all-weather track within a wider racecourse;
- `(July)` at Newmarket identifies a separately recognised racecourse.

Racecourse identity therefore requires an explicit governed mapping rather than a mechanical rule that strips parenthetical text.

In [6]:
from inside_rails.study_overlay import build_race_overlay_query


paired_labels = [
    "Kempton",
    "Kempton (AW)",
    "Lingfield",
    "Lingfield (AW)",
    "Newcastle",
    "Newcastle (AW)",
    "Newmarket",
    "Newmarket (July)",
    "Southwell",
    "Southwell (AW)",
]

placeholders = ", ".join("?" for _ in paired_labels)

# The accepted V3 race view contains race_type_raw.
# The project study overlay adds race_type_study so that any verified
# post-V3 race-type corrections are used without modifying the release.
base_query = f"""
SELECT
    raw_date,
    raw_course,
    raw_off,
    candidate_jurisdiction,
    candidate_course_label,
    race_type_raw,
    advertised_start_course_local
FROM view_reconciled_race_occurrences
WHERE candidate_jurisdiction = 'Great Britain'
  AND candidate_course_label IN ({placeholders})
"""

RESOLUTION_REGISTER = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "post_v3_external_value_resolutions.csv"
)

overlay_query, overlay_params = build_race_overlay_query(
    base_query,
    RESOLUTION_REGISTER,
)

# Aggregate the governed study-facing race type produced by the overlay.
query = f"""
SELECT
    candidate_course_label,
    race_type_study,
    COUNT(*) AS races
FROM ({overlay_query})
GROUP BY
    candidate_course_label,
    race_type_study
ORDER BY
    candidate_course_label,
    race_type_study
"""

with connect_read_only(DATABASE_PATH) as connection:
    paired_label_racing = pd.read_sql_query(
        query,
        connection,
        params=[*paired_labels, *overlay_params],
    )

paired_label_racing

,candidate_course_label,race_type_study,races
0,Kempton,Chase,361
1,Kempton,Hurdle,483
2,Kempton,NH Flat,58
3,Kempton (AW),Flat,5039
4,Kempton (AW),NH Flat,7
5,Lingfield,Chase,197
6,Lingfield,Flat,992
7,Lingfield,Hurdle,265
8,Lingfield,NH Flat,3
9,Lingfield (AW),Flat,4726


### What we found

The racing composition of the paired labels supports the conclusion that the source labels encode different kinds of distinctions.

At Kempton and Southwell, the unqualified labels are used for National Hunt racing while the `(AW)` labels are overwhelmingly used for Flat racing.

Newcastle shows a similar pattern: most racing under `Newcastle` is National Hunt, while almost all racing under `Newcastle (AW)` is Flat. A smaller population of Flat races also appears under the unqualified Newcastle label and should not be interpreted without further historical investigation.

Lingfield is particularly useful because the unqualified `Lingfield` label contains substantial Flat racing as well as Jump racing. This is consistent with Lingfield having a turf Flat course as well as an all-weather track. The distinction is therefore not simply `Flat versus Jump`; it is a distinction between racing configurations within the same racecourse.

Newmarket behaves differently again. Both `Newmarket` and `Newmarket (July)` contain only Flat racing, consistent with the external evidence that the Rowley Mile and July Course are separately recognised racecourses rather than alternative surfaces within one racecourse.

### Data issue to retain

Small numbers of races classified as `NH Flat` occur under several `(AW)` labels.

We should not infer an explanation from the label alone. These observations should remain visible and be investigated separately if they become material to the racecourse-identity decision.

### Interpretation

The evidence now shows that `candidate_course_label` mixes more than one kind of real-world entity distinction.

Some labels distinguish tracks or surfaces within one racecourse, while others distinguish separately recognised racecourses.

A mechanical transformation such as removing `(AW)` or stripping all parenthetical qualifiers would therefore be wrong.

In [7]:
# Show the complete British source-label inventory so we can check for
# aliases or historical naming variants that would not be caught by the
# parenthetical-label diagnostic.

pd.set_option("display.max_rows", None)

gb_course_labels[
    [
        "candidate_course_label",
        "races",
        "race_dates",
        "earliest_date",
        "latest_date",
    ]
]

,candidate_course_label,races,race_dates,earliest_date,latest_date
0,Aintree,610,87,2015-04-09,2026-05-15
1,Ascot,1816,273,2015-01-17,2026-05-09
2,Ayr,2325,328,2015-01-02,2026-05-20
3,Bangor-on-Dee,985,145,2015-01-06,2026-05-23
4,Bath,1540,216,2015-04-17,2026-05-26
5,Beverley,1444,199,2015-04-15,2026-05-27
6,Brighton,1484,214,2015-04-21,2025-10-16
7,Carlisle,1775,250,2015-02-16,2026-05-18
8,Cartmel,639,92,2015-05-25,2026-05-27
9,Catterick,2010,287,2015-01-01,2026-05-21


### Reconciliation with the official British racecourse count

The 65 distinct source course labels do not represent 65 racecourses.

Five groups of source labels refer to multiple courses, tracks or surfaces within a single reader-facing racecourse identity:

- `Kempton` and `Kempton (AW)` → Kempton Park
- `Lingfield` and `Lingfield (AW)` → Lingfield Park
- `Newcastle` and `Newcastle (AW)` → Newcastle
- `Newmarket` and `Newmarket (July)` → Newmarket
- `Southwell` and `Southwell (AW)` → Southwell

The British Horseracing Authority itself lists Newmarket as a single racecourse entry containing the July and Rowley Mile courses. This resolves the apparent tension between Newmarket having two separately named racing courses and its treatment in the national racecourse inventory.

Collapsing these five duplicated racecourse identities reduces the 65 source labels to **60 racecourses represented in the database period**.

One of those 60 is Towcester, whose final races in this dataset occurred in 2018. The BHA subsequently confirmed that Towcester Racecourse closed.

Excluding that historical racecourse leaves **59 currently operating racecourse identities**, matching the BHA's published total of 59 British racecourses.

### Interpretation

This provides an external reconciliation of the source-data mapping.

The database does not contain 65 British racecourses. It contains 65 source course labels representing 60 racecourses across the study period, of which 59 remain part of the current British racecourse population.

For reader-facing work, racecourse identity and course/track identity should therefore remain separate concepts.

In [8]:
# Explicit reader-facing racecourse identity mapping.
#
# Most source labels already correspond one-to-one with a racecourse.
# Only the known multi-label cases are collapsed deliberately.

racecourse_identity_map = {
    "Kempton": "Kempton Park",
    "Kempton (AW)": "Kempton Park",
    "Lingfield": "Lingfield Park",
    "Lingfield (AW)": "Lingfield Park",
    "Newcastle": "Newcastle",
    "Newcastle (AW)": "Newcastle",
    "Newmarket": "Newmarket",
    "Newmarket (July)": "Newmarket",
    "Southwell": "Southwell",
    "Southwell (AW)": "Southwell",
}

gb_racecourses = gb_course_labels.copy()

gb_racecourses["racecourse_identity"] = (
    gb_racecourses["candidate_course_label"]
    .map(racecourse_identity_map)
    .fillna(gb_racecourses["candidate_course_label"])
)

summary = pd.DataFrame(
    {
        "source_course_labels": [
            gb_racecourses["candidate_course_label"].nunique()
        ],
        "racecourse_identities_in_period": [
            gb_racecourses["racecourse_identity"].nunique()
        ],
        "current_racecourse_identities": [
            gb_racecourses.loc[
                gb_racecourses["racecourse_identity"] != "Towcester",
                "racecourse_identity",
            ].nunique()
        ],
    }
)

summary

,source_course_labels,racecourse_identities_in_period,current_racecourse_identities
0,65,60,59


### Racecourse count

The source data contains **65 distinct British course labels**, but these do not correspond one-to-one with racecourses.

After explicitly reconciling the known multi-label cases, those 65 labels represent **60 racecourse identities** across the study period.

Towcester is included historically in the dataset but is no longer an operating racecourse. Excluding it leaves **59 current British racecourse identities**.

This matches the British Horseracing Authority's published current total.

### Answer to the study question

A British racecourse should be counted as the recognised racing venue or racecourse identity, not as every distinct source label, track, surface or named course configuration within that venue.

The source data therefore requires a racecourse-identity mapping before it can be used to count British racecourses reliably.

In [9]:
# Show the final provisional racecourse identities and the source labels
# that contribute to each one. This is the last identity-level audit before
# the mapping is treated as a candidate for database promotion.

racecourse_identity_audit = (
    gb_racecourses
    .groupby("racecourse_identity", as_index=False)
    .agg(
        source_labels=(
            "candidate_course_label",
            lambda values: ", ".join(sorted(values)),
        ),
        source_label_count=("candidate_course_label", "nunique"),
        races=("races", "sum"),
        earliest_date=("earliest_date", "min"),
        latest_date=("latest_date", "max"),
    )
    .sort_values("racecourse_identity")
    .reset_index(drop=True)
)

racecourse_identity_audit

,racecourse_identity,source_labels,source_label_count,races,earliest_date,latest_date
0,Aintree,Aintree,1,610,2015-04-09,2026-05-15
1,Ascot,Ascot,1,1816,2015-01-17,2026-05-09
2,Ayr,Ayr,1,2325,2015-01-02,2026-05-20
3,Bangor-on-Dee,Bangor-on-Dee,1,985,2015-01-06,2026-05-23
4,Bath,Bath,1,1540,2015-04-17,2026-05-26
5,Beverley,Beverley,1,1444,2015-04-15,2026-05-27
6,Brighton,Brighton,1,1484,2015-04-21,2025-10-16
7,Carlisle,Carlisle,1,1775,2015-02-16,2026-05-18
8,Cartmel,Cartmel,1,639,2015-05-25,2026-05-27
9,Catterick,Catterick,1,2010,2015-01-01,2026-05-21


In [10]:
candidate_name_review = racecourse_identity_audit[
    racecourse_identity_audit["racecourse_identity"].isin(
        [
            "Chelmsford (AW)",
            "Epsom",
            "Kempton Park",
            "Lingfield Park",
            "Newcastle",
            "Newmarket",
            "Sandown",
            "Wolverhampton (AW)",
        ]
    )
].copy()

candidate_name_review

,racecourse_identity,source_labels,source_label_count,races,earliest_date,latest_date
10,Chelmsford (AW),Chelmsford (AW),1,4310,2015-01-11,2026-03-26
15,Epsom,Epsom,1,717,2015-04-22,2026-04-28
27,Kempton Park,"Kempton, Kempton (AW)",2,5948,2015-01-07,2026-05-27
29,Lingfield Park,"Lingfield, Lingfield (AW)",2,6260,2015-01-03,2026-05-26
34,Newcastle,"Newcastle, Newcastle (AW)",2,5416,2015-01-03,2026-05-19
35,Newmarket,"Newmarket, Newmarket (July)",2,2941,2015-04-15,2026-05-16
44,Sandown,Sandown,1,1687,2015-01-03,2026-04-25
56,Wolverhampton (AW),Wolverhampton (AW),1,7042,2015-01-02,2026-05-18


## Canonical racecourse naming

The source course label is not necessarily the canonical name of the racecourse.

Some differences are obvious — for example `Chelmsford (AW)` versus Chelmsford City, or `Sandown` versus Sandown Park — but apparently straightforward labels should not be assumed to be canonical merely because they resemble the commonly used name.

Before racecourse identities are promoted into the governed database, every identified racecourse will therefore be checked against authoritative racing sources and assigned an explicit canonical name.

Canonical names are attributes of stable racecourse identities; they are not themselves identifiers.

In [11]:
# Canonical reader-facing racecourse names verified against the BHA
# racecourse inventory. Towcester is retained as a historical identity
# because it occurs within the study period.

canonical_racecourse_names = {
    "Aintree": "Aintree",
    "Ascot": "Ascot",
    "Ayr": "Ayr",
    "Bangor-on-Dee": "Bangor-on-Dee",
    "Bath": "Bath",
    "Beverley": "Beverley",
    "Brighton": "Brighton",
    "Carlisle": "Carlisle",
    "Cartmel": "Cartmel",
    "Catterick": "Catterick",
    "Chelmsford (AW)": "Chelmsford City",
    "Cheltenham": "Cheltenham",
    "Chepstow": "Chepstow",
    "Chester": "Chester",
    "Doncaster": "Doncaster",
    "Epsom": "Epsom Downs",
    "Exeter": "Exeter",
    "Fakenham": "Fakenham",
    "Ffos Las": "Ffos Las",
    "Fontwell": "Fontwell Park",
    "Goodwood": "Goodwood",
    "Hamilton": "Hamilton Park",
    "Haydock": "Haydock Park",
    "Hereford": "Hereford",
    "Hexham": "Hexham",
    "Huntingdon": "Huntingdon",
    "Kelso": "Kelso",
    "Kempton Park": "Kempton Park",
    "Leicester": "Leicester",
    "Lingfield Park": "Lingfield Park",
    "Ludlow": "Ludlow",
    "Market Rasen": "Market Rasen",
    "Musselburgh": "Musselburgh",
    "Newbury": "Newbury",
    "Newcastle": "Newcastle",
    "Newmarket": "Newmarket",
    "Newton Abbot": "Newton Abbot",
    "Nottingham": "Nottingham",
    "Perth": "Perth",
    "Plumpton": "Plumpton",
    "Pontefract": "Pontefract",
    "Redcar": "Redcar",
    "Ripon": "Ripon",
    "Salisbury": "Salisbury",
    "Sandown": "Sandown Park",
    "Sedgefield": "Sedgefield",
    "Southwell": "Southwell",
    "Stratford": "Stratford-on-Avon",
    "Taunton": "Taunton",
    "Thirsk": "Thirsk",
    "Towcester": "Towcester",
    "Uttoxeter": "Uttoxeter",
    "Warwick": "Warwick",
    "Wetherby": "Wetherby",
    "Wincanton": "Wincanton",
    "Windsor": "Windsor",
    "Wolverhampton (AW)": "Wolverhampton",
    "Worcester": "Worcester",
    "Yarmouth": "Great Yarmouth",
    "York": "York",
}

racecourse_identity_audit["canonical_racecourse_name"] = (
    racecourse_identity_audit["racecourse_identity"]
    .map(canonical_racecourse_names)
)

# Fail closed: every racecourse identity must have an explicit
# verified canonical name before database promotion.
assert racecourse_identity_audit["canonical_racecourse_name"].notna().all()
assert len(canonical_racecourse_names) == 60

racecourse_identity_audit[
    [
        "racecourse_identity",
        "canonical_racecourse_name",
        "source_labels",
        "source_label_count",
    ]
]

,racecourse_identity,canonical_racecourse_name,source_labels,source_label_count
0,Aintree,Aintree,Aintree,1
1,Ascot,Ascot,Ascot,1
2,Ayr,Ayr,Ayr,1
3,Bangor-on-Dee,Bangor-on-Dee,Bangor-on-Dee,1
4,Bath,Bath,Bath,1
5,Beverley,Beverley,Beverley,1
6,Brighton,Brighton,Brighton,1
7,Carlisle,Carlisle,Carlisle,1
8,Cartmel,Cartmel,Cartmel,1
9,Catterick,Catterick,Catterick,1


In [12]:
# Final integrity checks for the candidate racecourse mapping.
#
# These checks make sure:
# - all 65 source labels are represented;
# - they resolve to exactly 60 racecourse identities;
# - every identity has exactly one canonical name;
# - canonical names are unique across those identities.

source_labels_covered = set()

for labels in racecourse_identity_audit["source_labels"]:
    source_labels_covered.update(label.strip() for label in labels.split(","))

assert source_labels_covered == set(gb_course_labels["candidate_course_label"])
assert racecourse_identity_audit["racecourse_identity"].nunique() == 60
assert racecourse_identity_audit["canonical_racecourse_name"].notna().all()
assert racecourse_identity_audit["canonical_racecourse_name"].nunique() == 60

mapping_integrity_summary = pd.DataFrame(
    {
        "source_labels": [len(source_labels_covered)],
        "racecourse_identities": [
            racecourse_identity_audit["racecourse_identity"].nunique()
        ],
        "canonical_names": [
            racecourse_identity_audit["canonical_racecourse_name"].nunique()
        ],
        "unmapped_source_labels": [
            len(
                set(gb_course_labels["candidate_course_label"])
                - source_labels_covered
            )
        ],
    }
)

mapping_integrity_summary

,source_labels,racecourse_identities,canonical_names,unmapped_source_labels
0,65,60,60,0


## Database implication

This study establishes that a source course label is not always the same thing as a racecourse identity.

For Great Britain, 65 distinct source course labels in the study period resolve to 60 racecourse identities. Some racecourses therefore require multiple source labels to map to the same stable entity.

The governed database should preserve both levels:

- the original or governed source course label, because it can carry analytically useful distinctions such as all-weather versus other racing configurations;
- a stable racecourse identity, used when the analytical question concerns the recognised racing venue rather than the source label;
- a canonical racecourse name as an attribute of that stable identity.

Racecourse identity should not encode temporary operational, licensing or current-status information. Those are time-dependent attributes and should be modelled separately if required.

The Great Britain mapping validated in this study should therefore become governed reference data rather than remain as notebook-only transformation logic.

## Existing database model

Database v3 does not currently document a stable governed racecourse identity.

The normal race-level analytical interface exposes `candidate_course_label`, which this study has shown can represent a finer distinction than racecourse identity.

Existing governed identity work covers other entity types, including horse occurrences and participant-label identities, but there is no equivalent documented racecourse-identity layer.

The result of this study therefore represents a new governed entity requirement rather than a reinterpretation of an existing racecourse identifier.

Before implementation, the database design should preserve the existing source/course-label level and add racecourse identity as a separate governed layer.

In [13]:
# Produce the explicit source-course-label → racecourse mapping that this
# study has validated. This is the candidate governed reference data that
# can later be promoted into the database.

source_to_racecourse_mapping = (
    gb_racecourses[
        [
            "candidate_course_label",
            "racecourse_identity",
        ]
    ]
    .merge(
        racecourse_identity_audit[
            [
                "racecourse_identity",
                "canonical_racecourse_name",
            ]
        ],
        on="racecourse_identity",
        how="left",
        validate="many_to_one",
    )
    .sort_values("candidate_course_label")
    .reset_index(drop=True)
)

assert len(source_to_racecourse_mapping) == 65
assert source_to_racecourse_mapping["canonical_racecourse_name"].notna().all()

source_to_racecourse_mapping

,candidate_course_label,racecourse_identity,canonical_racecourse_name
0,Aintree,Aintree,Aintree
1,Ascot,Ascot,Ascot
2,Ayr,Ayr,Ayr
3,Bangor-on-Dee,Bangor-on-Dee,Bangor-on-Dee
4,Bath,Bath,Bath
5,Beverley,Beverley,Beverley
6,Brighton,Brighton,Brighton
7,Carlisle,Carlisle,Carlisle
8,Cartmel,Cartmel,Cartmel
9,Catterick,Catterick,Catterick


### Explicit source-course mapping

The table above records the complete Great Britain mapping established by this study.

All 65 source course labels are assigned to one of 60 racecourse identities. The mapping preserves source-level distinctions where they exist while also identifying the racecourse to which each label belongs.

The important many-to-one mappings are:

- `Kempton` and `Kempton (AW)` → Kempton Park
- `Lingfield` and `Lingfield (AW)` → Lingfield Park
- `Newcastle` and `Newcastle (AW)` → Newcastle
- `Newmarket` and `Newmarket (July)` → Newmarket
- `Southwell` and `Southwell (AW)` → Southwell

Other source labels may differ from the canonical racecourse name without representing a second identity. Examples include:

- `Chelmsford (AW)` → Chelmsford City
- `Epsom` → Epsom Downs
- `Fontwell` → Fontwell Park
- `Hamilton` → Hamilton Park
- `Haydock` → Haydock Park
- `Sandown` → Sandown Park
- `Stratford` → Stratford-on-Avon
- `Wolverhampton (AW)` → Wolverhampton
- `Yarmouth` → Great Yarmouth

This explicit mapping, rather than a string-cleaning rule, is the candidate reference data for later database governance.

## What characteristics define a racecourse and its courses?

Having established racecourse identity, the next question is what stable physical and racing characteristics should be associated with that identity.

Some characteristics apply to the racecourse as a whole, while others describe a particular course or track within it.

Candidate characteristics include:

- racing surface;
- synthetic surface type where applicable;
- direction or handedness;
- racing disciplines supported;
- presence of separate straight and round courses;
- course layout and configuration;
- circuit length and other stable physical dimensions;
- relevant jumps-track characteristics.

These characteristics will be investigated before deciding the governed database model.

## Surface is a course-level and potentially time-dependent characteristic

Surface cannot safely be treated as a single permanent attribute of a racecourse.

A racecourse may contain more than one racing surface. For example:

- Lingfield Park has both turf and all-weather racing tracks;
- Newcastle has a turf jumps track and a separate all-weather Flat track.

The type of synthetic surface can also change over time. Southwell's all-weather track, for example, changed from Fibresand to Tapeta during the study period.

This means the eventual governed model needs to distinguish:

1. the racecourse;
2. the particular course or track within that racecourse;
3. the surface used by that course or track;
4. where necessary, the dates for which that surface definition applies.

Surface should therefore not be stored only as a permanent racecourse-level attribute.

### Working definition of a course / track

For this study, a **course or track** is a materially distinct racing route or surface configuration within a racecourse that is recognised separately for the staging of races.

A racecourse may therefore contain more than one course or track.

Examples include:

- separate turf and all-weather tracks;
- separately named Flat courses such as Newmarket's Rowley Mile and July Course;
- distinct hurdle and steeplechase courses where those use materially different racing routes;
- separate straight and round courses where they constitute distinct racing configurations.

Temporary running-rail movements or routine changes to the exact racing line do not by themselves create a new course identity.

Course characteristics such as surface, handedness and layout will be attached at this course/track level where applicable rather than assumed to describe the whole racecourse.

In [14]:
# Evidence register for racecourse/course characteristics.
#
# Every value that may later be promoted into the governed database should
# have explicit provenance rather than relying on notebook narrative alone.

provenance_columns = [
    "jurisdiction",
    "racecourse_identity",
    "course_or_track_name",
    "characteristic",
    "value",
    "valid_from",
    "valid_to",
    "source_authority",
    "source_title",
    "source_url",
    "accessed_date",
    "evidence_note",
    "verification_status",
]

course_characteristic_provenance = pd.DataFrame(
    columns=provenance_columns
)

course_characteristic_provenance

,jurisdiction,racecourse_identity,course_or_track_name,characteristic,value,valid_from,valid_to,source_authority,source_title,source_url,accessed_date,evidence_note,verification_status


### Provenance rule

Every racecourse or course characteristic intended for later database promotion must have explicit supporting evidence.

The evidence register records:

- the racecourse and course/track concerned;
- the characteristic and asserted value;
- any period for which that value is valid;
- the authority and source used;
- the source title and URL;
- the date the source was accessed;
- a concise note explaining what the evidence establishes;
- verification status.

No characteristic should be promoted into the governed database solely because it appears in notebook prose or because it is commonly assumed to be true.

Where a characteristic changes over time, separate evidence-backed records should be retained for each valid period.

## Aintree

The course-level review identifies three distinct racing courses at Aintree:

- **Grand National Course**
- **Mildmay Course**
- **Hurdle Course**

The Jockey Club treats these as distinct courses, including by publishing separate going information for the Grand National Course and the Mildmay and Hurdle Courses.

### Grand National Course

The Grand National Course is the specialist steeplechase course containing Aintree's distinctive Grand National fences.

The Jockey Club identifies 16 fences on this course, 14 of which are jumped twice during the Grand National.

### Mildmay Course

The Mildmay Course is Aintree's conventional steeplechase course.

Jockey Club race material explicitly identifies races such as the Grade 1 Bowl Chase as being staged on the Mildmay Course.

### Hurdle Course

The Hurdle Course is separately recognised from the Mildmay Course in official Aintree going reports and is used for hurdle racing.

### Surface

Aintree's racing courses are turf. Official racecourse welfare material describes the management of the turf and grass across the racing circuit and around its fences.

### Handedness

Aintree is left-handed. Jockey Club material explicitly describes the racecourse as left-handed.

### Course inventory

| Course / track | Surface | Primary use | Handedness |
|---|---|---|---|
| Grand National Course | Turf | Steeplechase / National fences | Left-handed |
| Mildmay Course | Turf | Conventional chase | Left-handed |
| Hurdle Course | Turf | Hurdle | Left-handed |

### Study implication

The single source label `Aintree` therefore represents at least three distinct governed course entities within one racecourse.

This is further evidence that source course labels cannot themselves serve as the governed course inventory.

In [17]:
aintree_provenance_rows = [
    {
        "jurisdiction": "Great Britain",
        "racecourse_identity": "Aintree",
        "course_or_track_name": "Grand National Course",
        "characteristic": "course_identity",
        "value": "Grand National Course",
        "valid_from": None,
        "valid_to": None,
        "source_authority": "The Jockey Club",
        "source_title": "Going news ahead of Randox Grand National Festival",
        "source_url": (
            "https://www.thejockeyclub.co.uk/aintree/media/press-releases/"
            "2022/04/going-news-ahead-of-randox-grand-national-festival-plus-"
            "best-shod-award-returns-and-randox-grand-national-trophy-revealed/"
        ),
        "accessed_date": "2026-08-10",
        "evidence_note": (
            "Official Aintree going report separately identifies the "
            "Grand National, Mildmay and Hurdle courses."
        ),
        "verification_status": "verified",
    },
    {
        "jurisdiction": "Great Britain",
        "racecourse_identity": "Aintree",
        "course_or_track_name": "Mildmay Course",
        "characteristic": "course_identity",
        "value": "Mildmay Course",
        "valid_from": None,
        "valid_to": None,
        "source_authority": "The Jockey Club",
        "source_title": "Going news ahead of Randox Grand National Festival",
        "source_url": (
            "https://www.thejockeyclub.co.uk/aintree/media/press-releases/"
            "2022/04/going-news-ahead-of-randox-grand-national-festival-plus-"
            "best-shod-award-returns-and-randox-grand-national-trophy-revealed/"
        ),
        "accessed_date": "2026-08-10",
        "evidence_note": (
            "Official Aintree going report separately identifies the "
            "Grand National, Mildmay and Hurdle courses."
        ),
        "verification_status": "verified",
    },
    {
        "jurisdiction": "Great Britain",
        "racecourse_identity": "Aintree",
        "course_or_track_name": "Hurdle Course",
        "characteristic": "course_identity",
        "value": "Hurdle Course",
        "valid_from": None,
        "valid_to": None,
        "source_authority": "The Jockey Club",
        "source_title": "Going news ahead of Randox Grand National Festival",
        "source_url": (
            "https://www.thejockeyclub.co.uk/aintree/media/press-releases/"
            "2022/04/going-news-ahead-of-randox-grand-national-festival-plus-"
            "best-shod-award-returns-and-randox-grand-national-trophy-revealed/"
        ),
        "accessed_date": "2026-08-10",
        "evidence_note": (
            "Official Aintree going report separately identifies the "
            "Grand National, Mildmay and Hurdle courses."
        ),
        "verification_status": "verified",
    },
]

course_characteristic_provenance = pd.concat(
    [
        course_characteristic_provenance,
        pd.DataFrame(aintree_provenance_rows),
    ],
    ignore_index=True,
)

## Ascot

The course-level review identifies three materially distinct racing configurations at Ascot:

- **Straight Course**
- **Round Course**
- **National Hunt / Jumps Course**

### Straight Course

The BHA explicitly recognises Ascot's Straight Course separately from the Round Course.

Flat races can be programmed entirely on the Straight Course when the Round Course is unavailable. The straight configuration is used for shorter Flat races and includes Ascot's straight mile.

Because it contains no racing bend, handedness is not intrinsically meaningful for a race run wholly on the Straight Course. The racecourse-level BHA classification of Ascot as right-handed is based on races involving the major bend.

### Round Course

Ascot's Round Course is a distinct Flat racing configuration.

The BHA explicitly distinguishes it from the Straight Course and classifies Ascot as a **right-handed** Flat racecourse because the major bend used in races around a turn is right-handed.

### National Hunt / Jumps Course

Ascot also stages National Hunt racing, including hurdle and steeplechase races.

Available course-layout evidence identifies a separate National Hunt racing circuit from the Flat configurations. It is a right-handed course.

At this stage the evidence reviewed does not justify splitting the National Hunt circuit further into permanent separate `Hurdle Course` and `Chase Course` identities merely because different obstacle types are placed on it.

### Surface

The racing configurations reviewed are turf.

There is no evidence of an all-weather racing course at Ascot during the study period.

### Course inventory

| Course / track | Surface | Primary use | Handedness |
|---|---|---|---|
| Straight Course | Turf | Flat | Not applicable — straight |
| Round Course | Turf | Flat | Right-handed |
| National Hunt / Jumps Course | Turf | Hurdle / Chase | Right-handed |

### Study implication

Ascot demonstrates that handedness should not simply be copied from the racecourse onto every constituent course.

The BHA classifies Ascot as right-handed because of the direction of its major bend, but a race run wholly on the Straight Course has no bend and therefore does not itself have a meaningful left- or right-handed direction.

Handedness consequently belongs at course/configuration level where applicable, with an explicit `not applicable` state available for genuinely straight courses.

In [18]:
ascot_provenance_rows = [
    {
        "jurisdiction": "Great Britain",
        "racecourse_identity": "Ascot",
        "course_or_track_name": "Straight Course",
        "characteristic": "course_identity",
        "value": "Straight Course",
        "valid_from": None,
        "valid_to": None,
        "source_authority": "British Horseracing Authority",
        "source_title": (
            "ASCOT, MAY 11TH AND 12TH: EXTRA RACE AND DIVISIONS "
            "TO SAFEGUARD RACING"
        ),
        "source_url": (
            "https://www.britishhorseracing.com/press_releases/"
            "ascot-may-11th-and-12th-extra-race-and-divisions-to-safeguard-racing/"
        ),
        "accessed_date": "2026-08-10",
        "evidence_note": (
            "BHA explicitly distinguishes Ascot's straight course from "
            "its round course and describes races being staged on it."
        ),
        "verification_status": "verified",
    },
    {
        "jurisdiction": "Great Britain",
        "racecourse_identity": "Ascot",
        "course_or_track_name": "Round Course",
        "characteristic": "course_identity",
        "value": "Round Course",
        "valid_from": None,
        "valid_to": None,
        "source_authority": "British Horseracing Authority",
        "source_title": (
            "ASCOT, MAY 11TH AND 12TH: EXTRA RACE AND DIVISIONS "
            "TO SAFEGUARD RACING"
        ),
        "source_url": (
            "https://www.britishhorseracing.com/press_releases/"
            "ascot-may-11th-and-12th-extra-race-and-divisions-to-safeguard-racing/"
        ),
        "accessed_date": "2026-08-10",
        "evidence_note": (
            "BHA explicitly distinguishes Ascot's round course from "
            "its straight course."
        ),
        "verification_status": "verified",
    },
    {
        "jurisdiction": "Great Britain",
        "racecourse_identity": "Ascot",
        "course_or_track_name": "Round Course",
        "characteristic": "handedness",
        "value": "right-handed",
        "valid_from": None,
        "valid_to": None,
        "source_authority": "British Horseracing Authority",
        "source_title": "NEW RANGE OF FLAT RACE STARTING PROCEDURES TO BE TRIALLED",
        "source_url": (
            "https://www.britishhorseracing.com/press_releases/"
            "new-range-of-flat-race-starting-procedures-to-be-trialled/"
        ),
        "accessed_date": "2026-08-10",
        "evidence_note": (
            "BHA classifies Ascot as right-handed based on the major bend "
            "when races are run around a turn."
        ),
        "verification_status": "verified",
    },
    {
        "jurisdiction": "Great Britain",
        "racecourse_identity": "Ascot",
        "course_or_track_name": "Straight Course",
        "characteristic": "handedness",
        "value": "not_applicable",
        "valid_from": None,
        "valid_to": None,
        "source_authority": "British Horseracing Authority",
        "source_title": "NEW RANGE OF FLAT RACE STARTING PROCEDURES TO BE TRIALLED",
        "source_url": (
            "https://www.britishhorseracing.com/press_releases/"
            "new-range-of-flat-race-starting-procedures-to-be-trialled/"
        ),
        "accessed_date": "2026-08-10",
        "evidence_note": (
            "BHA states its right-handed classification is based on the "
            "major bend used when racing around a turn; a wholly straight "
            "course has no handed bend."
        ),
        "verification_status": "verified_interpretation",
    },
]

course_characteristic_provenance = pd.concat(
    [
        course_characteristic_provenance,
        pd.DataFrame(ascot_provenance_rows),
    ],
    ignore_index=True,
)

## Change in research structure

The course-level investigation showed that each racecourse can require substantial venue-specific research.

Rather than expanding this cross-racecourse study indefinitely, detailed course research will now be maintained in **one notebook per racecourse**.

Each racecourse notebook will act as the durable research record for that venue and can be revisited as later studies establish additional characteristics or analytical findings.

The present study remains responsible for:

- defining racecourse and course/track identity;
- maintaining the Great Britain racecourse inventory;
- maintaining the source-label → racecourse mapping;
- establishing which constituent courses/tracks exist;
- drawing cross-racecourse conclusions.

Individual racecourse notebooks will hold the detailed supporting research and provenance.

Initial venue notebooks:

- `Aintree`
- `Ascot`

The Aintree and Ascot research already completed in this study should be transferred into those venue notebooks rather than duplicated here in full.